# 30. Did unlearning shrink the weights, or scramble them?

Unlearning drives the generated weights toward their offsets. The 21 October measurement
found that 41 percent of channels still respond afterwards, at roughly a sixth of their
earlier strength, and 59 percent went silent. That tells us the weights got smaller. It
does not tell us whether the ones that survived still point the same way.

Those are two very different outcomes:

* **Same direction, smaller.** The pattern survived and only its size changed. Multiplying
  the weights back up should restore accuracy with no training. The task is recoverable,
  and a privacy claim based on this unlearning is weak.
* **Different direction.** The pattern is gone. Multiplying up would amplify noise. Nothing
  is recoverable by rescaling.

This notebook decides between them with a dot product. No training, no images, no GPU.
It reads two saved models and compares the weights they generate for one task.

Run this before any rescaling experiment. If the directions do not survive, the rescaling
experiment is already answered and does not need to run.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
# Importing uncle twice in one session gives two copies of the same classes, and a
# checkpoint saved by one will not load into the other. Stop before that happens.
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")
if IN_COLAB:
    REPO = Path("/content/path-direction")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")
# boto3 talks to R2. tqdm is pulled in by uncle.telemetry.
for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.measure import relative_change

import torch
# Everything here is a dot product over saved tensors. A GPU would sit idle.
DEVICE = "cpu"
torch.set_grad_enabled(False)

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| device:", DEVICE)

## 2. Settings

`FORGET_TASK` is the task that was unlearned. `CONTROL_TASK` is a task the same model kept.
The control matters: it says whether anything we see is specific to the forgotten task or
happened to the whole generator. On 21 October the kept task lost nothing measurable, so we
expect its cosine to sit at 1.0. If it does not, something is wrong with the comparison
rather than with the model.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"
ORDER = "order_01"
SEEDS = [0, 1, 2]

# The 21 October neuron measurement used task 3 as the forgotten task and task 0 as the
# kept control. Keeping the same pair makes the two measurements comparable.
FORGET_TASK = "3"
CONTROL_TASK = "0"

# A channel counts as dead when its weights have essentially no size left. Direction has no
# meaning for a zero vector, so these are reported separately rather than averaged in.
DEAD_NORM = 1e-8
# Below this cosine we treat a channel's direction as changed rather than preserved.
# 0.9 is about 26 degrees. It is a reporting threshold, not a statistical test.
PRESERVED_ABOVE = 0.9

# The same cache notebook 29 uses, so a source model downloaded by either is reused.
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
RESULT_PATH = CACHE / f"direction_{STUDY_ID}_{ORDER}.json"
SAVE_TO_R2 = True
RESULT_KEY = f"uncle/unlearning_direction/{STUDY_ID}/{ORDER}/cosines.json"

print("Forgotten task:", FORGET_TASK, "| control task:", CONTROL_TASK)
print("Seeds:", SEEDS)

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
PREFIX = "uncle/paired_generalization/" + STUDY_ID
plan = json.loads(store.download(PREFIX + "/study.json", CACHE / "cache").read_text())
storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = list(plan["orders"][ORDER])
print("Study:", plan["study_id"], "| order", ORDER, "learns", ORDER_TASKS)

## 4. See which model pairs exist

We need two models that differ only by the unlearning step.

* **Before**: the source model, which learned all eight tasks. Sources are kept permanently,
  so this one is always available.
* **After**: the same model immediately after unlearning the forgotten task, before it
  learned anything new.

The after model is the one that may be missing. The study plan sets
`keep_post_unlearning_checkpoint` to false by default, which means those models were deleted
once their reports were verified. This cell checks rather than assumes, and prints exactly
what it found. If nothing is available, stop here: producing a new unlearned model needs a
GPU, and that is a separate job.

In [ ]:
# Branch job ids look like branches/<order>/seed_<n>/forget_<task>/learn_<new>. We want any
# branch that forgot FORGET_TASK, because the post-unlearning model is taken before the
# branch learns its new task, so every branch with the same forget task gives the same model.
def branches_for(seed):
    base = f"{ORDER}/seed_{seed}"
    return [job for job in ALL_JOBS
            if job["id"].startswith(f"branches/{base}/forget_{FORGET_TASK}/")]

def source_job(seed):
    return next(job for job in ALL_JOBS if job["id"] == f"sources/{ORDER}/seed_{seed}")

available, missing = [], []
for seed in SEEDS:
    # The source model: present, finished, and still holding its checkpoint.
    try:
        source_progress = storage.progress(source_job(seed))
        has_source = source_progress is not None and source_progress["status"] == "complete"
    except Exception as problem:
        has_source, source_progress = False, None
        print(f"seed {seed}: source unreadable: {problem}")
    # The post-unlearning model is recorded inside the branch's completed report, not in the
    # progress pointer, so we read the report and look for that artifact.
    found_after = None
    for job in branches_for(seed):
        try:
            report = storage.completed_report(job)
        except Exception as problem:
            print(f"seed {seed}: {job['id']} report unreadable: {problem}")
            continue
        if report and report.get("post_unlearning_checkpoint"):
            found_after = (job["id"], report["post_unlearning_checkpoint"])
            break
    if has_source and found_after:
        available.append((seed, found_after[0], found_after[1]))
        print(f"seed {seed}: ready, after-model from {found_after[0]}")
    else:
        missing.append(seed)
        why = []
        if not has_source:
            why.append("no finished source model")
        if not found_after:
            why.append("no retained post-unlearning model")
        print(f"seed {seed}: not ready ({', '.join(why)})")

print()
print(f"{len(available)} of {len(SEEDS)} seeds can be measured.")
if not available:
    print("Nothing to measure. The post-unlearning models were not kept.")
    print("To produce one, rerun a branch with keep_post_unlearning_checkpoint set to true.")
    print("That needs a GPU and is not part of this notebook.")

## 5. The measurement

For one task, the generator produces a full set of classifier weights. We take that set from
the before model and from the after model, and ask how much the direction changed.

**Cosine similarity** between two lists of numbers is their dot product divided by both
their lengths. It ignores size completely and reports only direction:

* 1.0 means the same direction. The weights were scaled, not changed.
* 0.0 means unrelated directions.
* -1.0 means the opposite direction.

Two details matter for reading the number correctly.

**We subtract the offsets first.** Each generated weight is a raw generator output times a
fixed scale, plus a fixed offset. BatchNorm scale weights have an offset of 1.0. If we left
it in, a weight that collapsed entirely to 1.0 would still look like a perfect match, because
the constant would dominate. Subtracting the offset leaves the part unlearning actually acts
on.

**We measure per output channel as well as per tensor.** A whole-tensor cosine averages over
every channel at once, so a few preserved channels can hide many scrambled ones. The 21
October finding was that 59 percent of channels went silent, which is a per-channel fact, so
we report the per-channel distribution too.

In [ ]:
def cosine(before, after):
    """Direction agreement between two tensors, ignoring their size.

    Returns NaN when either side has no length, because a zero vector points nowhere and
    any number we invented for it would be read as a result.
    """
    a = before.double().reshape(-1)
    b = after.double().reshape(-1)
    size_a, size_b = a.norm(), b.norm()
    if size_a <= DEAD_NORM or size_b <= DEAD_NORM:
        return float("nan")
    return float((a @ b) / (size_a * size_b))


def channel_report(before, after):
    """The same question asked separately for each output channel.

    Row 0 of a weight tensor is one output channel: one convolution filter, or one row of
    the classifier. Flattening everything after dimension 0 gives one vector per channel.
    Tensors with a single dimension, such as biases, have no channel structure, so they
    return None and are covered by the whole-tensor cosine instead.
    """
    if before.dim() < 2:
        return None
    a = before.double().reshape(before.shape[0], -1)
    b = after.double().reshape(after.shape[0], -1)
    size_a, size_b = a.norm(dim=1), b.norm(dim=1)
    alive = (size_a > DEAD_NORM) & (size_b > DEAD_NORM)
    cosines = ((a * b).sum(dim=1)[alive] / (size_a[alive] * size_b[alive]))
    return {
        "channels": int(before.shape[0]),
        # Channels that had size before and have none now. Rescaling cannot bring these
        # back: zero times any multiplier is still zero.
        "died": int(((size_a > DEAD_NORM) & (size_b <= DEAD_NORM)).sum()),
        "measurable": int(alive.sum()),
        "median_cosine": float(cosines.median()) if cosines.numel() else float("nan"),
        "lowest_cosine": float(cosines.min()) if cosines.numel() else float("nan"),
        "preserved": int((cosines >= PRESERVED_ABOVE).sum()),
        # How much smaller each surviving channel became, as a typical value. If unlearning
        # is a pure scaling, this is the number a rescaling experiment would multiply by.
        "median_size_ratio": float((size_b[alive] / size_a[alive]).median()) if alive.any() else float("nan"),
    }


def generated_part(hypernet, weights):
    """Remove each tensor's fixed offset, leaving what the generator actually produced."""
    return {name: value - hypernet.offsets[name] for name, value in weights.items()}


def build_hypernet(seed, payload):
    """Rebuild the generator from a saved checkpoint, with no data and no trainer.

    The task codes are a ParameterDict, so the entries have to exist before the saved state
    can be loaded into them. We create them in the order the checkpoint recorded, then load.
    Nothing here touches images, which is why this notebook needs no dataset.
    """
    config = Config(**{**plan["config"], "seed": seed, "device": DEVICE,
                       "requests": tuple(("learn", task) for task in ORDER_TASKS)})
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    for task in payload["tasks_with_codes"]:
        hypernet.add_task(task)
    hypernet.load_state_dict(payload["hypernet"])
    hypernet.eval()
    return hypernet

## 6. Run it

One seed at a time, so only two models are ever in memory. For each seed we measure the
forgotten task and the control task with exactly the same code, because a comparison is only
as trustworthy as its control.

In [ ]:
results = []
for seed, branch_id, artifact in available:
    print(f"seed {seed}: loading models")
    before_payload = storage.load(source_job(seed))
    after_path = store.download(artifact["key"], CACHE / "cache")
    after_payload = torch.load(after_path, map_location="cpu", weights_only=False)

    before_net = build_hypernet(seed, before_payload)
    after_net = build_hypernet(seed, after_payload)

    for task, role in ((FORGET_TASK, "forgotten"), (CONTROL_TASK, "control")):
        if task not in before_payload["tasks_with_codes"]:
            print(f"  task {task} has no code in the before model, skipped")
            continue
        before_weights = generated_part(before_net, before_net.weights_for(task))
        after_weights = generated_part(after_net, after_net.weights_for(task))

        # One number for the whole task: the generator's raw output before scaling. This is
        # the quantity the unlearning objective pushes toward zero, so it is the most direct
        # answer to whether the direction survived.
        raw_cosine = cosine(before_net.raw_for(task), after_net.raw_for(task))
        # How much the generated part moved in total, relative to its own earlier size.
        # This is the magnitude half of the picture, and cosine is the direction half.
        # Note this runs on the offset-subtracted weights, so it is not the same number as
        # a relative change computed on the full weights.
        moved = relative_change(after_weights, before_weights)

        per_tensor = {}
        for name in sorted(before_weights):
            per_tensor[name] = {
                "cosine": cosine(before_weights[name], after_weights[name]),
                "offset": float(before_net.offsets[name]),
                "channels": channel_report(before_weights[name], after_weights[name]),
            }
        results.append({"seed": seed, "task": task, "role": role, "branch": branch_id,
                        "raw_cosine": raw_cosine, "relative_change": moved,
                        "per_tensor": per_tensor})
        print(f"  task {task} ({role}): raw cosine {raw_cosine:.4f}, "
              f"relative change {moved:.4f}")
    del before_net, after_net, before_payload, after_payload

RESULT_PATH.write_text(json.dumps(
    {"head": HEAD, "study": STUDY_ID, "order": ORDER, "forget_task": FORGET_TASK,
     "control_task": CONTROL_TASK, "results": results}, indent=2), encoding="utf-8")
print("\nSaved", RESULT_PATH)
if SAVE_TO_R2 and results:
    # overwrite, because rerunning this notebook should replace its own result rather
    # than stop with an error.
    store.upload(RESULT_PATH, RESULT_KEY, overwrite=True)
    print("Uploaded", RESULT_KEY)

## 7. Read the result

The summary below is per seed and task. Read the forgotten row against the control row.

In [ ]:
print(f"{'seed':>4} {'task':>5} {'role':>10} {'raw cos':>9} {'rel chg':>9} "
      f"{'med cos':>9} {'died':>7} {'kept dir':>9}")
for entry in results:
    # Pool the per-channel numbers across tensors so one line describes the whole model.
    reports = [t["channels"] for t in entry["per_tensor"].values() if t["channels"]]
    died = sum(r["died"] for r in reports)
    measurable = sum(r["measurable"] for r in reports)
    preserved = sum(r["preserved"] for r in reports)
    medians = [r["median_cosine"] for r in reports if r["median_cosine"] == r["median_cosine"]]
    median = sorted(medians)[len(medians) // 2] if medians else float("nan")
    share = f"{100.0 * preserved / measurable:.0f}%" if measurable else "n/a"
    print(f"{entry['seed']:>4} {entry['task']:>5} {entry['role']:>10} "
          f"{entry['raw_cosine']:>9.4f} {entry['relative_change']:>9.4f} "
          f"{median:>9.4f} {died:>7} {share:>9}")

## What the numbers mean

Read the forgotten task's row, then check it against the control row.

**Control row.** Its raw cosine should sit at 1.0000 and its relative change at 0.0000. The
control task was not unlearned, and the generator's weights for it were not reported to move
on 21 October. If this row is not near those values, the two models differ for some other
reason and the forgotten row cannot be trusted yet.

**Forgotten task, high cosine, high relative change.** The pattern survived and only its size
changed. This supports the rescaling experiment: multiply the generated weights back up and
check whether accuracy returns with no training. It also means a privacy claim for this
unlearning is weak, because the information is still present, just quiet.

**Forgotten task, low cosine.** The pattern changed, not just its size. Rescaling would
amplify something different from the original weights, so the rescaling experiment is
answered and does not need GPU time.

**The died column.** Channels that had size before and have none now. Rescaling cannot
recover these, because any multiplier times zero is still zero. If this number is large while
the surviving cosines are high, unlearning did two different things at once: it removed some
channels and it turned others down. Those two need separate conclusions.

**median size ratio, in the saved JSON.** If unlearning is close to a pure scaling, this is
the factor a rescaling experiment should multiply by, measured per tensor. Compare it with
the per-stage values from 21 October: 0.29 at the stem, then 0.14, 0.13, 0.16 and 0.19. Those
came from channel responses to images, not from the weights, so agreement between the two is
a check and a disagreement is worth understanding before rescaling anything.

## What this notebook does not do

It does not measure effective rank, which needs a forward pass over images. It does not test
recovery; it only says whether a recovery attempt by rescaling is worth running. And it reads
whatever seeds happen to have a retained post-unlearning model, so the number of seeds here
may be smaller than three. Say how many when reporting any of this.